# Sixpack_Chroma 파라미터별 정확도 비교 평가

재계산된 Sixpack_Chroma 벡터들을 로드하여 분류 정확도를 비교합니다.

## 평가 파이프라인
1. 각 파라미터 조건별 벡터 로드
2. PCA 차원 축소 (20D / 50D / 100D / No PCA)
3. 분류기 평가 (SVM, KNN, RF + Soft Margin SVM)
4. Soft/Strict Accuracy 비교
5. 비교 테이블 및 시각화

## 1. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Colab)
matplotlib.rcParams['axes.unicode_minus'] = False

print("All imports loaded.")

## 2. Ground Truth & Adjacent Phases

In [ ]:
# Phase Diagram에서 직접 추출한 인접 관계 (GROUND_TRUTH_M 기준, 27쌍)
# 최종.ipynb final code와 동일한 기준
# NOTE: 논문 정의에 있던 phase 5, 8은 실제 GT에 존재하지 않으므로 제외
def extract_adjacent_phases(matrices):
    adjacent_pairs = set()
    for M in matrices:
        M = np.array(M)
        rows, cols = M.shape
        for i in range(rows):
            for j in range(cols):
                current = M[i, j]
                neighbors = []
                if i > 0: neighbors.append(M[i-1, j])
                if i < rows-1: neighbors.append(M[i+1, j])
                if j > 0: neighbors.append(M[i, j-1])
                if j < cols-1: neighbors.append(M[i, j+1])
                for neighbor in neighbors:
                    if current != neighbor:
                        pair = tuple(sorted([int(current), int(neighbor)]))
                        adjacent_pairs.add(pair)
    adj_dict = {}
    for p1, p2 in adjacent_pairs:
        adj_dict.setdefault(p1, []).append(p2)
        adj_dict.setdefault(p2, []).append(p1)
    return adj_dict

ADJACENT_PHASES = extract_adjacent_phases(GROUND_TRUTH_M)

def are_adjacent_phases(label1, label2, adjacent_dict=None):
    if adjacent_dict is None:
        adjacent_dict = ADJACENT_PHASES
    label1, label2 = int(label1), int(label2)
    if label1 == label2:
        return True
    if label1 in adjacent_dict and label2 in adjacent_dict[label1]:
        return True
    if label2 in adjacent_dict and label1 in adjacent_dict[label2]:
        return True
    return False

def soft_accuracy_score(y_true, y_pred, ignore_adjacent=True, adjacent_dict=None):
    if adjacent_dict is None:
        adjacent_dict = ADJACENT_PHASES
    if not ignore_adjacent:
        return accuracy_score(y_true, y_pred)
    n_correct = 0
    n_total = len(y_true)
    for true_label, pred_label in zip(y_true, y_pred):
        if true_label == pred_label:
            n_correct += 1
        elif are_adjacent_phases(int(true_label), int(pred_label), adjacent_dict):
            n_correct += 1
    return n_correct / n_total if n_total > 0 else 0.0

print(f"Classes: {np.unique(GROUND_TRUTH_M)} ({len(np.unique(GROUND_TRUTH_M))} classes)")
print(f"ADJACENT_PHASES (phase diagram 추출, 최종.ipynb 기준): {len(ADJACENT_PHASES)} entries")
for phase in sorted(ADJACENT_PHASES.keys()):
    print(f"  Phase {phase:2d} <-> {sorted(ADJACENT_PHASES[phase])}")

## 3. 데이터 로딩 함수

In [ ]:
def load_sixpack_chroma_data(data_dir):
    """
    Sixpack_Chroma 벡터 로드.
    npz 파일 구조: arr_0 = PI_A_to_B (dict), arr_1 = PI_B_to_A (dict)
    각 dict: {barcode_type: {0: H0_vector, 1: H1_vector}}
    """
    npz_files = sorted(glob.glob(os.path.join(data_dir, "Sixpack_Chroma_*.npz")))
    if len(npz_files) == 0:
        return None, None, None

    X_list, y_list, indices = [], [], []

    for filepath in npz_files:
        filename = os.path.basename(filepath)
        try:
            sim_idx = int(filename.split('_')[-1].split('.')[0])
            label = get_label_from_index(sim_idx)
            data = np.load(filepath, allow_pickle=True)

            arr_0 = data['arr_0'].item() if hasattr(data['arr_0'], 'item') else data['arr_0']
            arr_1 = data['arr_1'].item() if hasattr(data['arr_1'], 'item') else data['arr_1']

            features = []
            if isinstance(arr_0, dict):
                for key in sorted(arr_0.keys()):
                    val = arr_0[key]
                    if isinstance(val, dict):
                        for dim_key in sorted(val.keys()):
                            v = np.array(val[dim_key]) if not isinstance(val[dim_key], np.ndarray) else val[dim_key]
                            features.extend(v.flatten())
                    else:
                        val = np.array(val) if not isinstance(val, np.ndarray) else val
                        features.extend(val.flatten())
                for key in sorted(arr_1.keys()):
                    val = arr_1[key]
                    if isinstance(val, dict):
                        for dim_key in sorted(val.keys()):
                            v = np.array(val[dim_key]) if not isinstance(val[dim_key], np.ndarray) else val[dim_key]
                            features.extend(v.flatten())
                    else:
                        val = np.array(val) if not isinstance(val, np.ndarray) else val
                        features.extend(val.flatten())
            else:
                arr_0 = np.array(arr_0) if not isinstance(arr_0, np.ndarray) else arr_0
                arr_1 = np.array(arr_1) if not isinstance(arr_1, np.ndarray) else arr_1
                features.extend(arr_0.flatten())
                features.extend(arr_1.flatten())

            X_list.append(features)
            y_list.append(label)
            indices.append(sim_idx)
        except Exception as e:
            print(f"  Error: {filename} - {e}")

    if len(X_list) == 0:
        return None, None, None

    return np.nan_to_num(np.array(X_list), nan=0.0, posinf=0.0, neginf=0.0), np.array(y_list), indices

print("데이터 로딩 함수 정의 완료.")

## 4. 분류 평가 함수

In [ ]:
def evaluate_all_classifiers(X, y, n_splits=5, pca_dim=None, verbose=False):
    """
    여러 분류기로 평가. Soft/Strict accuracy + F1 score 모두 계산.

    Returns:
        dict: {classifier_name: {soft_mean, soft_std, strict_mean, strict_std, f1_mean, f1_std}}
    """
    # Scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # PCA
    actual_dim = X_scaled.shape[1]
    if pca_dim is not None and actual_dim > pca_dim:
        pca = PCA(n_components=pca_dim, random_state=42)
        X_reduced = pca.fit_transform(X_scaled)
    else:
        X_reduced = X_scaled
        pca_dim = actual_dim

    classifiers = {
        'KNN (k=3)': KNeighborsClassifier(n_neighbors=3),
        'KNN (k=12)': KNeighborsClassifier(n_neighbors=12),
        'SVM (RBF, C=1)': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
        'SVM (Linear, C=1)': SVC(kernel='linear', C=1.0, random_state=42),
        'SVM (RBF, C=0.5)': SVC(kernel='rbf', C=0.5, gamma='scale', random_state=42),
        'SVM (RBF, C=2)': SVC(kernel='rbf', C=2.0, gamma='scale', random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    }

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = {}

    for clf_name, clf_template in classifiers.items():
        soft_accs, strict_accs, f1_scores = [], [], []

        for train_idx, test_idx in skf.split(X_reduced, y):
            from sklearn.base import clone
            clf = clone(clf_template)
            clf.fit(X_reduced[train_idx], y[train_idx])
            y_pred = clf.predict(X_reduced[test_idx])

            soft_accs.append(soft_accuracy_score(y[test_idx], y_pred, ignore_adjacent=True))
            strict_accs.append(accuracy_score(y[test_idx], y_pred))
            f1_scores.append(f1_score(y[test_idx], y_pred, average='weighted', zero_division=0))

        results[clf_name] = {
            'soft_mean': np.mean(soft_accs) * 100,
            'soft_std': np.std(soft_accs) * 100,
            'strict_mean': np.mean(strict_accs) * 100,
            'strict_std': np.std(strict_accs) * 100,
            'f1_mean': np.mean(f1_scores) * 100,
            'f1_std': np.std(f1_scores) * 100,
        }

        if verbose:
            r = results[clf_name]
            print(f"    {clf_name:25s} | Soft: {r['soft_mean']:.2f}±{r['soft_std']:.2f}% "
                  f"| Strict: {r['strict_mean']:.2f}±{r['strict_std']:.2f}% "
                  f"| F1: {r['f1_mean']:.2f}±{r['f1_std']:.2f}%")

    return results, pca_dim


def get_best_result(results):
    """결과 dict에서 가장 높은 soft accuracy를 가진 분류기 반환."""
    best_clf = max(results, key=lambda k: results[k]['soft_mean'])
    return best_clf, results[best_clf]


print("분류 평가 함수 정의 완료.")

## 5-6. 실험 데이터 순차 로드 & 평가 (메모리 절약)

각 실험 조건을 **한 번에 하나씩** 로드→평가→삭제하여 RAM을 절약합니다.
(alpha20: 242,400D ≈ 1GB/실험, 13개 동시 로드 시 ~10GB 필요)

In [ ]:
import gc

VECTOR_DIR = "/content/drive/MyDrive/URP/1224_Vectors"
PCA_DIM = 20
N_SPLITS = 5

all_chroma_dirs = sorted(glob.glob(os.path.join(VECTOR_DIR, "Sixpack_Chroma*")))
print(f"발견된 Sixpack_Chroma 디렉토리: {len(all_chroma_dirs)}개")
print("=" * 70)

all_results = {}   # 결과만 저장 (가벼움)
experiment_data = {}  # confusion matrix용: best 1개만 유지

for d in all_chroma_dirs:
    dir_name = os.path.basename(d)
    n_files = len(glob.glob(os.path.join(d, "Sixpack_Chroma_*.npz")))
    if n_files == 0:
        print(f"  ✗ {dir_name}: 파일 없음")
        continue

    print(f"\n[{dir_name}] ({n_files} files) loading...", end=" ")
    X, y, idx = load_sixpack_chroma_data(d)
    if X is None:
        print("✗ 로드 실패")
        continue

    print(f"Shape: {X.shape} → evaluating...", end=" ")
    results, actual_dim = evaluate_all_classifiers(
        X, y, n_splits=N_SPLITS, pca_dim=PCA_DIM, verbose=False
    )

    best_clf, best_scores = get_best_result(results)
    print(f"✓ Best: {best_clf} = {best_scores['soft_mean']:.2f}% (soft)")

    all_results[dir_name] = {
        'classifiers': results,
        'original_dim': X.shape[1],
        'pca_dim': actual_dim,
        'n_samples': X.shape[0],
    }

    # Best 실험만 데이터 유지 (confusion matrix용)
    current_best_soft = best_scores['soft_mean']
    prev_best_soft = 0
    if experiment_data:
        prev_key = list(experiment_data.keys())[0]
        prev_best = get_best_result(all_results[prev_key]['classifiers'])
        prev_best_soft = prev_best[1]['soft_mean']

    if current_best_soft > prev_best_soft:
        experiment_data.clear()
        experiment_data[dir_name] = {'X': X, 'y': y, 'indices': idx}
    else:
        del X, y, idx

    gc.collect()

print("\n" + "=" * 70)
print(f"✓ 전체 {len(all_results)}개 실험 평가 완료!")
print(f"  메모리에 유지된 데이터: {list(experiment_data.keys())}")
print("=" * 70)


## 7. 결과 비교 테이블

In [ ]:
# ============================================
# 7-1. Best Classifier 요약 테이블
# ============================================

summary_rows = []

for exp_name, result in all_results.items():
    best_clf, best_scores = get_best_result(result['classifiers'])
    summary_rows.append({
        'Experiment': exp_name,
        'Dim': result['original_dim'],
        'Best Classifier': best_clf,
        'Soft Acc (%)': f"{best_scores['soft_mean']:.2f} ± {best_scores['soft_std']:.2f}",
        'Strict Acc (%)': f"{best_scores['strict_mean']:.2f} ± {best_scores['strict_std']:.2f}",
        'F1 (%)': f"{best_scores['f1_mean']:.2f} ± {best_scores['f1_std']:.2f}",
        'soft_mean': best_scores['soft_mean'],  # for sorting
    })

df_summary = pd.DataFrame(summary_rows)
df_summary = df_summary.sort_values('soft_mean', ascending=False)
df_display = df_summary.drop(columns=['soft_mean'])

print("\n" + "=" * 110)
print("Best Classifier per Experiment (sorted by Soft Accuracy)")
print(f"PCA: {PCA_DIM}D, {N_SPLITS}-fold CV")
print("=" * 110)
print(df_display.to_string(index=False))
print("=" * 110)

In [ ]:
# ============================================
# 7-2. 전체 분류기별 상세 테이블
# ============================================

# 가장 성능 좋은 분류기 2개 + SVM Linear 고정
HIGHLIGHT_CLFS = ['SVM (Linear, C=1)', 'SVM (RBF, C=1)', 'KNN (k=3)']

detail_rows = []
for exp_name, result in all_results.items():
    for clf_name in HIGHLIGHT_CLFS:
        if clf_name in result['classifiers']:
            r = result['classifiers'][clf_name]
            detail_rows.append({
                'Experiment': exp_name,
                'Classifier': clf_name,
                'Soft Acc': r['soft_mean'],
                'Strict Acc': r['strict_mean'],
                'F1': r['f1_mean'],
            })

df_detail = pd.DataFrame(detail_rows)

# Pivot table: experiments × classifiers
for metric in ['Soft Acc', 'Strict Acc', 'F1']:
    pivot = df_detail.pivot(index='Experiment', columns='Classifier', values=metric)
    pivot = pivot.sort_values(by=pivot.columns[0], ascending=False)
    print(f"\n{'='*80}")
    print(f"{metric} (%) - Comparison")
    print(f"{'='*80}")
    print(pivot.to_string(float_format="%.2f"))
    print()

## 8. 시각화

In [ ]:
# ============================================
# 8-1. Soft Accuracy Bar Chart (비교)
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Sort experiments by best soft accuracy
exp_order = df_summary.sort_values('soft_mean', ascending=True)['Experiment'].tolist()

for ax_idx, metric_name in enumerate(['Soft Acc', 'Strict Acc']):
    ax = axes[ax_idx]

    # Collect data for each classifier
    for clf_name in HIGHLIGHT_CLFS:
        values = []
        for exp_name in exp_order:
            if exp_name in all_results and clf_name in all_results[exp_name]['classifiers']:
                key = 'soft_mean' if metric_name == 'Soft Acc' else 'strict_mean'
                values.append(all_results[exp_name]['classifiers'][clf_name][key])
            else:
                values.append(0)

        y_pos = np.arange(len(exp_order))
        offset = (HIGHLIGHT_CLFS.index(clf_name) - 1) * 0.25
        ax.barh(y_pos + offset, values, 0.22, label=clf_name, alpha=0.85)

    ax.set_yticks(np.arange(len(exp_order)))
    ax.set_yticklabels([e.replace('Sixpack_Chroma_', '') for e in exp_order], fontsize=9)
    ax.set_xlabel(f'{metric_name} (%)')
    ax.set_title(f'{metric_name} by Experiment')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim([0, 100])

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/URP/sixpack_chroma_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

In [ ]:
# ============================================
# 8-2. Heatmap: Experiments × Classifiers
# ============================================

# Build matrix for heatmap
all_clfs = ['KNN (k=3)', 'KNN (k=12)', 'SVM (RBF, C=0.5)', 'SVM (RBF, C=1)',
            'SVM (RBF, C=2)', 'SVM (Linear, C=1)', 'Random Forest']

# Use soft accuracy
heatmap_data = []
exp_labels = []
for exp_name in exp_order:
    row = []
    for clf_name in all_clfs:
        if clf_name in all_results[exp_name]['classifiers']:
            row.append(all_results[exp_name]['classifiers'][clf_name]['soft_mean'])
        else:
            row.append(0)
    heatmap_data.append(row)
    exp_labels.append(exp_name.replace('Sixpack_Chroma_', '').replace('Sixpack_Chroma', 'baseline_orig'))

heatmap_arr = np.array(heatmap_data)

fig, ax = plt.subplots(figsize=(14, max(6, len(exp_labels) * 0.5)))
im = ax.imshow(heatmap_arr, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(np.arange(len(all_clfs)))
ax.set_xticklabels(all_clfs, rotation=45, ha='right', fontsize=9)
ax.set_yticks(np.arange(len(exp_labels)))
ax.set_yticklabels(exp_labels, fontsize=9)

# Annotate with values
for i in range(len(exp_labels)):
    for j in range(len(all_clfs)):
        val = heatmap_arr[i, j]
        color = 'white' if val < 40 or val > 85 else 'black'
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, label='Soft Accuracy (%)', shrink=0.8)
ax.set_title('Soft Accuracy Heatmap: Experiment × Classifier')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/URP/sixpack_chroma_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Heatmap saved.")

## 9. PCA 차원별 비교 (Optional)

최상위 3개 조건에 대해 PCA 차원을 변경하며 정확도를 비교합니다.

In [ ]:
# Top 3 실험 조건 선택
top3_exps = df_summary.head(3)['Experiment'].tolist()
PCA_DIMS = [10, 20, 50, 100, None]  # None = No PCA
TARGET_CLF = 'SVM (Linear, C=1)'

pca_comparison = {}

print("PCA 차원별 비교")
print("=" * 70)

for exp_name in top3_exps:
    data = experiment_data[exp_name]
    print(f"\n[{exp_name}]")
    pca_comparison[exp_name] = {}

    for pca_dim in PCA_DIMS:
        dim_label = f"{pca_dim}D" if pca_dim else "NoPCA"
        results, actual_dim = evaluate_all_classifiers(
            data['X'], data['y'],
            n_splits=N_SPLITS,
            pca_dim=pca_dim,
            verbose=False
        )
        if TARGET_CLF in results:
            r = results[TARGET_CLF]
            pca_comparison[exp_name][dim_label] = r
            print(f"  PCA {dim_label:6s} → Soft: {r['soft_mean']:.2f}% | "
                  f"Strict: {r['strict_mean']:.2f}% | F1: {r['f1_mean']:.2f}%")

# PCA 비교 Plot
fig, ax = plt.subplots(figsize=(10, 6))
dim_labels = [f"{d}D" if d else "NoPCA" for d in PCA_DIMS]

for exp_name in top3_exps:
    soft_vals = [pca_comparison[exp_name].get(dl, {}).get('soft_mean', 0) for dl in dim_labels]
    label = exp_name.replace('Sixpack_Chroma_', '')
    ax.plot(dim_labels, soft_vals, 'o-', label=label, linewidth=2, markersize=8)

ax.set_xlabel('PCA Dimension')
ax.set_ylabel('Soft Accuracy (%)')
ax.set_title(f'PCA Dimension Impact ({TARGET_CLF})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/URP/sixpack_chroma_pca_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Confusion Matrix (Best Experiment)

In [ ]:
# Best experiment의 confusion matrix 생성
best_exp_name = df_summary.iloc[0]['Experiment']
best_data = experiment_data[best_exp_name]

print(f"Best experiment: {best_exp_name}")

X = best_data['X']
y = best_data['y']

# Scaling + PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
if X_scaled.shape[1] > PCA_DIM:
    pca = PCA(n_components=PCA_DIM, random_state=42)
    X_reduced = pca.fit_transform(X_scaled)
else:
    X_reduced = X_scaled

# Full training + prediction (for confusion matrix)
clf = SVC(kernel='linear', C=1.0, random_state=42)
clf.fit(X_reduced, y)
y_pred = clf.predict(X_reduced)

# Also collect cross-validated predictions
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = np.zeros_like(y)
for train_idx, test_idx in skf.split(X_reduced, y):
    clf_cv = SVC(kernel='linear', C=1.0, random_state=42)
    clf_cv.fit(X_reduced[train_idx], y[train_idx])
    y_pred_cv[test_idx] = clf_cv.predict(X_reduced[test_idx])

# Plot confusion matrix (CV predictions)
classes = sorted(np.unique(y))
cm = confusion_matrix(y, y_pred_cv, labels=classes)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Roman numeral labels
roman_map = {0: 'i', 1: 'ii', 2: 'iii', 3: 'iv', 4: 'v', 6: 'vi',
             7: 'vii', 8: 'viii', 9: 'ix', 10: 'x', 11: 'xi', 12: 'xii', 13: 'xiii'}
class_labels = [roman_map.get(c, str(c)) for c in classes]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Raw counts
im0 = axes[0].imshow(cm, cmap='Blues', interpolation='nearest')
axes[0].set_title(f'Confusion Matrix (Counts)\n{best_exp_name}')
axes[0].set_xticks(range(len(class_labels)))
axes[0].set_xticklabels(class_labels, rotation=45)
axes[0].set_yticks(range(len(class_labels)))
axes[0].set_yticklabels(class_labels)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

for i in range(len(class_labels)):
    for j in range(len(class_labels)):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8, color=color)

# Normalized
im1 = axes[1].imshow(cm_normalized, cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
axes[1].set_title(f'Confusion Matrix (Normalized)\n{best_exp_name}')
axes[1].set_xticks(range(len(class_labels)))
axes[1].set_xticklabels(class_labels, rotation=45)
axes[1].set_yticks(range(len(class_labels)))
axes[1].set_yticklabels(class_labels)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

for i in range(len(class_labels)):
    for j in range(len(class_labels)):
        val = cm_normalized[i, j]
        color = 'white' if val > 0.5 else 'black'
        axes[1].text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/URP/sixpack_chroma_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification Report
print("\nClassification Report (Cross-Validated):")
print(classification_report(y, y_pred_cv, target_names=class_labels, zero_division=0))

# Identity matrix similarity (Frobenius norm)
identity = np.eye(len(classes))
frob_dist = np.linalg.norm(cm_normalized - identity, 'fro')
print(f"\nFrobenius distance from identity: {frob_dist:.4f}")
print(f"(Lower = better, 0 = perfect classification)")

## 11. 결과 CSV 저장

In [ ]:
# 전체 결과를 CSV로 저장
csv_rows = []

for exp_name, result in all_results.items():
    for clf_name, scores in result['classifiers'].items():
        csv_rows.append({
            'Experiment': exp_name,
            'Original_Dim': result['original_dim'],
            'PCA_Dim': result['pca_dim'],
            'Classifier': clf_name,
            'Soft_Acc_Mean': scores['soft_mean'],
            'Soft_Acc_Std': scores['soft_std'],
            'Strict_Acc_Mean': scores['strict_mean'],
            'Strict_Acc_Std': scores['strict_std'],
            'F1_Mean': scores['f1_mean'],
            'F1_Std': scores['f1_std'],
        })

df_all = pd.DataFrame(csv_rows)
csv_path = '/content/drive/MyDrive/URP/sixpack_chroma_comparison_results.csv'
df_all.to_csv(csv_path, index=False)
print(f"결과 저장 완료: {csv_path}")
print(f"총 {len(csv_rows)}개 행 ({len(all_results)}개 실험 × {len(csv_rows)//len(all_results)}개 분류기)")

# 미리보기
print("\n미리보기 (Soft Accuracy 상위 10):")
print(df_all.sort_values('Soft_Acc_Mean', ascending=False).head(10).to_string(index=False))

## 12. 기존 방법들과의 비교

최종.ipynb 실측 결과 (Corrected Adjacent Phases, RBF kernel, 5-fold CV, PCA 20D) 기반.
새로운 Sixpack_Chroma 파라미터 실험 결과와 기존 7개 방법을 종합 비교합니다.

In [ ]:
# ============================================
# 섹션 12: 기존 방법들과의 비교
# 최종.ipynb 실측 결과 (Corrected Adjacent Phases)
# RBF kernel, 5-fold CV, PCA 20D
# ============================================
# 이 코드를 Sixpack_Chroma_비교평가.ipynb의 섹션 12 셀에 붙여넣으세요.

BASELINE_RESULTS = {
    'Inter_PI': {
        'dim': 10200,
        'KNN (k=3)':      {'soft': 95.89, 'strict': 67.96},
        'KNN (k=12)':     {'soft': 96.87, 'strict': 69.14},
        'SVM (RBF)':      {'soft': 93.35, 'strict': 58.98},
        'SVM (Linear)':   {'soft': 97.27, 'strict': 72.86},
        'Random Forest':  {'soft': 97.46, 'strict': 74.60},
        'S-SVM (C=0.5)':  {'soft': 93.55, 'strict': 59.36},
        'S-SVM (C=1.0)':  {'soft': 94.33, 'strict': 60.54},
        'S-SVM (C=2.0)':  {'soft': 95.70, 'strict': 64.45},
    },
    '3D_PI': {
        'dim': 16800,
        'KNN (k=3)':      {'soft': 93.95, 'strict': 60.93},
        'KNN (k=12)':     {'soft': 94.33, 'strict': 64.06},
        'SVM (RBF)':      {'soft': 73.44, 'strict': 25.98},
        'SVM (Linear)':   {'soft': 96.87, 'strict': 71.67},
        'Random Forest':  {'soft': 98.64, 'strict': 74.41},
        'S-SVM (C=0.5)':  {'soft': 78.71, 'strict': 30.65},
        'S-SVM (C=1.0)':  {'soft': 78.71, 'strict': 35.55},
        'S-SVM (C=2.0)':  {'soft': 84.17, 'strict': 40.03},
    },
    'Ord_PI': {
        'dim': 10200,
        'KNN (k=3)':      {'soft': 97.07, 'strict': 68.16},
        'KNN (k=12)':     {'soft': 98.04, 'strict': 70.11},
        'SVM (RBF)':      {'soft': 89.84, 'strict': 45.51},
        'SVM (Linear)':   {'soft': 96.88, 'strict': 73.64},
        'Random Forest':  {'soft': 97.66, 'strict': 74.61},
        'S-SVM (C=0.5)':  {'soft': 89.45, 'strict': 43.95},
        'S-SVM (C=1.0)':  {'soft': 91.99, 'strict': 47.26},
        'S-SVM (C=2.0)':  {'soft': 94.53, 'strict': 57.42},
    },
    'Sixpack_Chroma (기존)': {
        'dim': 61200,
        'KNN (k=3)':      {'soft': 95.51, 'strict': 70.13},
        'KNN (k=12)':     {'soft': 96.48, 'strict': 67.57},
        'SVM (RBF)':      {'soft': 79.50, 'strict': 38.67},
        'SVM (Linear)':   {'soft': 98.05, 'strict': 75.78},
        'Random Forest':  {'soft': 96.68, 'strict': 72.85},
        'S-SVM (C=0.5)':  {'soft': 78.91, 'strict': 37.50},
        'S-SVM (C=1.0)':  {'soft': 80.86, 'strict': 39.84},
        'S-SVM (C=2.0)':  {'soft': 84.18, 'strict': 42.77},
    },
    'Sixpack_Rips': {
        'dim': 288,
        'KNN (k=3)':      {'soft': 99.42, 'strict': 83.40},
        'KNN (k=12)':     {'soft': 99.61, 'strict': 80.47},
        'SVM (RBF)':      {'soft': 99.42, 'strict': 83.20},
        'SVM (Linear)':   {'soft': 99.61, 'strict': 85.35},
        'Random Forest':  {'soft': 99.81, 'strict': 84.77},
        'S-SVM (C=0.5)':  {'soft': 98.25, 'strict': 78.91},
        'S-SVM (C=1.0)':  {'soft': 99.22, 'strict': 82.81},
        'S-SVM (C=2.0)':  {'soft': 98.83, 'strict': 83.98},
    },
    'Inter+Ord': {
        'dim': 20400,
        'KNN (k=3)':      {'soft': 96.48, 'strict': 68.16},
        'KNN (k=12)':     {'soft': 97.26, 'strict': 68.16},
        'SVM (RBF)':      {'soft': 90.04, 'strict': 45.31},
        'SVM (Linear)':   {'soft': 97.27, 'strict': 75.79},
        'Random Forest':  {'soft': 96.68, 'strict': 73.82},
        'S-SVM (C=0.5)':  {'soft': 90.23, 'strict': 44.92},
        'S-SVM (C=1.0)':  {'soft': 90.43, 'strict': 46.29},
        'S-SVM (C=2.0)':  {'soft': 92.19, 'strict': 55.46},
    },
    '3D_Ord': {
        'dim': 27000,
        'KNN (k=3)':      {'soft': 96.68, 'strict': 65.62},
        'KNN (k=12)':     {'soft': 96.49, 'strict': 68.17},
        'SVM (RBF)':      {'soft': 73.05, 'strict': 26.56},
        'SVM (Linear)':   {'soft': 97.85, 'strict': 74.61},
        'Random Forest':  {'soft': 98.04, 'strict': 76.38},
        'S-SVM (C=0.5)':  {'soft': 73.44, 'strict': 26.17},
        'S-SVM (C=1.0)':  {'soft': 76.95, 'strict': 27.15},
        'S-SVM (C=2.0)':  {'soft': 77.34, 'strict': 37.12},
    },
}

print("기존 최종.ipynb 결과 로드 완료.")
print(f"  {len(BASELINE_RESULTS)}개 방법, 각 8개 분류기")

In [ ]:
# ============================================
# 12-1. 분류기별 종합 비교 테이블
# ============================================

COMPARE_CLFS = ['SVM (Linear)', 'Random Forest', 'KNN (k=3)', 'S-SVM (C=2.0)']

best_exp_name = df_summary.iloc[0]['Experiment']
best_new_results = all_results[best_exp_name]['classifiers']

clf_map = {
    'SVM (Linear)': 'SVM (Linear, C=1)',
    'Random Forest': 'Random Forest',
    'KNN (k=3)': 'KNN (k=3)',
    'S-SVM (C=2.0)': 'SVM (RBF, C=2)',
}

for clf_label in COMPARE_CLFS:
    print(f"\n{'='*100}")
    print(f"  {clf_label} — Soft/Strict Accuracy (PCA 20D, 5-fold CV)")
    print(f"{'='*100}")
    print(f"  {'Method':30s} {'Dim':>8s} {'Soft Acc':>12s} {'Strict Acc':>12s}")
    print(f"  {'-'*66}")

    rows = []
    for method, data in BASELINE_RESULTS.items():
        if clf_label in data:
            s = data[clf_label]
            rows.append((method, data['dim'], s['soft'], s['strict']))

    new_clf_key = clf_map.get(clf_label)
    if new_clf_key and new_clf_key in best_new_results:
        r = best_new_results[new_clf_key]
        rows.append((f'★ NEW {best_exp_name}',
                     all_results[best_exp_name]['original_dim'],
                     r['soft_mean'], r['strict_mean']))

    rows.sort(key=lambda x: x[2], reverse=True)
    for method, dim, soft, strict in rows:
        marker = '  ' if not method.startswith('★') else '→ '
        print(f"{marker}{method:30s} {dim:>8d} {soft:>10.2f}% {strict:>10.2f}%")
    print(f"  {'='*66}")

In [ ]:
# ============================================
# 12-2. 종합 비교 시각화 (Bar Chart)
# ============================================

method_best = {}
for method, data in BASELINE_RESULTS.items():
    best_soft, best_strict, best_clf = 0, 0, ''
    for clf_name, scores in data.items():
        if clf_name == 'dim':
            continue
        if scores['soft'] > best_soft:
            best_soft = scores['soft']
            best_strict = scores['strict']
            best_clf = clf_name
    method_best[method] = {'soft': best_soft, 'strict': best_strict, 'clf': best_clf}

for i in range(min(3, len(df_summary))):
    exp_name = df_summary.iloc[i]['Experiment']
    exp_results = all_results[exp_name]['classifiers']
    best_clf = max(exp_results, key=lambda k: exp_results[k]['soft_mean'])
    method_best[f'NEW: {exp_name}'] = {
        'soft': exp_results[best_clf]['soft_mean'],
        'strict': exp_results[best_clf]['strict_mean'],
        'clf': best_clf,
    }

sorted_methods = sorted(method_best.items(), key=lambda x: x[1]['soft'])
methods = [m[0] for m in sorted_methods]
soft_vals = [m[1]['soft'] for m in sorted_methods]
strict_vals = [m[1]['strict'] for m in sorted_methods]

colors_soft = ['#2196F3' if not m.startswith('NEW:') else '#FF5722' for m in methods]
colors_strict = ['#64B5F6' if not m.startswith('NEW:') else '#FF8A65' for m in methods]

fig, ax = plt.subplots(figsize=(12, max(6, len(methods) * 0.55)))
y_pos = np.arange(len(methods))

bars1 = ax.barh(y_pos + 0.15, soft_vals, 0.3, color=colors_soft, alpha=0.9, label='Soft Acc (adj-tol)')
bars2 = ax.barh(y_pos - 0.15, strict_vals, 0.3, color=colors_strict, alpha=0.9, label='Strict Acc')

clean_labels = []
for m in methods:
    label = m.replace('Sixpack_Chroma_', 'SC_').replace('NEW: Sixpack_Chroma_', '★ SC_')
    label = label.replace('NEW: ', '★ ')
    clean_labels.append(label)

ax.set_yticks(y_pos)
ax.set_yticklabels(clean_labels, fontsize=9)
ax.set_xlabel('Accuracy (%)')
ax.set_title('All Methods: Best Classifier Accuracy (PCA 20D, 5-fold CV)\n'
             'Blue = Baseline, Red = New Sixpack_Chroma Experiments')
ax.legend(loc='lower right')
ax.set_xlim([0, 105])
ax.grid(axis='x', alpha=0.3)

for bar, val in zip(bars1, soft_vals):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
            va='center', fontsize=8)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/URP/all_methods_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

In [ ]:
# ============================================
# 12-3. Improvement Summary
# ============================================

old_chroma = BASELINE_RESULTS['Sixpack_Chroma (기존)']
best_exp_name = df_summary.iloc[0]['Experiment']

print("=" * 90)
print("Sixpack_Chroma 개선 효과 요약")
print("=" * 90)
print(f"Best new experiment: {best_exp_name}")
print()

clf_map_rev = {
    'SVM (Linear, C=1)': 'SVM (Linear)',
    'SVM (RBF, C=1)': 'SVM (RBF)',
    'KNN (k=3)': 'KNN (k=3)',
    'Random Forest': 'Random Forest',
}

print(f"  {'Classifier':25s} {'Old Soft':>10s} {'New Soft':>10s} {'Δ Soft':>8s} | "
      f"{'Old Strict':>12s} {'New Strict':>12s} {'Δ Strict':>10s}")
print(f"  {'-'*95}")

for new_clf, old_clf in clf_map_rev.items():
    if new_clf in all_results[best_exp_name]['classifiers'] and old_clf in old_chroma:
        new_r = all_results[best_exp_name]['classifiers'][new_clf]
        old_r = old_chroma[old_clf]
        d_soft = new_r['soft_mean'] - old_r['soft']
        d_strict = new_r['strict_mean'] - old_r['strict']
        sign_s = '+' if d_soft >= 0 else ''
        sign_st = '+' if d_strict >= 0 else ''
        print(f"  {old_clf:25s} {old_r['soft']:>9.2f}% {new_r['soft_mean']:>9.2f}% {sign_s}{d_soft:>6.2f}% | "
              f"{old_r['strict']:>10.2f}% {new_r['strict_mean']:>10.2f}% {sign_st}{d_strict:>8.2f}%")

print(f"  {'='*95}")
print()

# Sixpack_Rips와의 격차
rips = BASELINE_RESULTS['Sixpack_Rips']
new_best_linear = all_results[best_exp_name]['classifiers'].get('SVM (Linear, C=1)', {})
rips_linear = rips.get('SVM (Linear)', {})

if new_best_linear and rips_linear:
    gap = rips_linear['soft'] - new_best_linear.get('soft_mean', 0)
    print(f"Sixpack_Rips (SVM Linear) soft = {rips_linear['soft']:.2f}%")
    print(f"New Sixpack_Chroma (SVM Linear) soft = {new_best_linear.get('soft_mean', 0):.2f}%")
    print(f"Gap: {gap:.2f}% (closer to Rips = better)")